In [4]:
!pip install scikit-image

In [ ]:
#!/usr/bin/env python
import pickle
import sys
import os
import numpy as np
import nibabel as nib
import networkx as nx
from scipy.ndimage import convolve, label, distance_transform_edt

#Needs Python 3.12.4

try:
    import torch
    from torch_geometric.data import Data
    PYG_AVAILABLE = True
except Exception:
    PYG_AVAILABLE = False


# ---------------------------------------------------------
# Skeletonization (3D)
# ---------------------------------------------------------
def skeletonize_3d_safe(vol):
    try:
        from skimage.morphology import skeletonize_3d
        return skeletonize_3d(vol.astype(bool)).astype(np.uint8)
    except ImportError:
        from skimage.morphology import skeletonize
        skel = np.zeros_like(vol, dtype=bool)
        for z in range(vol.shape[0]):
            skel[z] = skeletonize(vol[z])
        return skel.astype(np.uint8)


# ---------------------------------------------------------
# Compute degree (26-connected)
# ---------------------------------------------------------
def compute_degree_fast(skel):
    kernel = np.ones((3, 3, 3), dtype=np.uint8)
    kernel[1, 1, 1] = 0
    neighbors = convolve(skel.astype(np.uint8), kernel)
    return neighbors * skel

# ---------------------------------------------------------
# Curvature
# ---------------------------------------------------------
def compute_curvature(branch_voxels):
    pts = branch_voxels.astype(float)
    if len(pts) < 3:
        return 0.0

    diffs = np.diff(pts, axis=0)
    norms = np.linalg.norm(diffs, axis=1, keepdims=True) + 1e-8
    tangents = diffs / norms

    dots = np.sum(tangents[:-1] * tangents[1:], axis=1)
    dots = np.clip(dots, -1.0, 1.0)
    angles = np.arccos(dots)
    return float(np.mean(angles))


# ---------------------------------------------------------
# Tortuosity
# ---------------------------------------------------------
def compute_tortuosity(branch_voxels):
    K = len(branch_voxels)
    if K < 2:
        return 1.0
    path_len = float(K)
    d = np.linalg.norm(branch_voxels[-1] - branch_voxels[0]) + 1e-8
    return float(path_len / d)


# ---------------------------------------------------------
# Radius via distance transform
# ---------------------------------------------------------
def compute_branch_radius(branch_voxels, edt):
    Z, Y, X = edt.shape
    vals = []
    for (z, y, x) in branch_voxels:
        z, y, x = int(z), int(y), int(x)
        if 0 <= z < Z and 0 <= y < Y and 0 <= x < X:
            vals.append(edt[z, y, x])
    if len(vals) == 0:
        return 0.0
    return float(np.mean(vals))


# ---------------------------------------------------------
# Compute radius, curvature, tortuosity for each branch
# ---------------------------------------------------------
def compute_branch_features(branches, seg_full):
    """
    seg_full: full segmentation (can be multiclass).
              Radius uses seg_full > 0 for EDT.
    """
    edt = distance_transform_edt(seg_full > 0)
    out = []
    for br in branches:
        vox = br["voxels"]
        rad = compute_branch_radius(vox, edt)
        curv = compute_curvature(vox)
        tort = compute_tortuosity(vox)
        out.append({
            "node_u": br["node_u"],
            "node_v": br["node_v"],
            "voxels": vox,
            "length_vox": float(len(vox)),
            "radius": rad,
            "curvature": curv,
            "tortuosity": tort
        })
    return out


# ---------------------------------------------------------
# Extract branches
# ---------------------------------------------------------
def extract_branches_fast(skel, nodes_mask, links_mask):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    labeled_links, num = label(links_mask, structure)

    node_positions = np.column_stack(np.where(nodes_mask))
    node_tuples = [tuple(p) for p in node_positions]
    node_to_id = {p: i for i, p in enumerate(node_tuples)}

    Z, Y, X = skel.shape
    neighbors26 = np.array(
        [[dz, dy, dx]
         for dz in (-1, 0, 1)
         for dy in (-1, 0, 1)
         for dx in (-1, 0, 1)
         if not (dz == 0 and dy == 0 and dx == 0)],
        dtype=int
    )

    branches = []

    for comp in range(1, num + 1):
        coords = np.column_stack(np.where(labeled_links == comp))
        # coords shape: (N, 3)

        # Find neighboring node voxels
        nbrs = coords[:, None, :] + neighbors26[None, :, :]

        valid = (
            (nbrs[..., 0] >= 0) & (nbrs[..., 0] < Z) &
            (nbrs[..., 1] >= 0) & (nbrs[..., 1] < Y) &
            (nbrs[..., 2] >= 0) & (nbrs[..., 2] < X)
        )
        nbr_valid = nbrs[valid]

        touching = nodes_mask[nbr_valid[:, 0],
                              nbr_valid[:, 1],
                              nbr_valid[:, 2]]
        node_vox = nbr_valid[touching]

        if len(node_vox) < 2:
            continue

        un = np.unique(node_vox, axis=0)

        if len(un) == 2:
            u, v = un
        else:
            dif = un[:, None, :] - un[None, :, :]
            dst = np.linalg.norm(dif, axis=2)
            i, j = np.unravel_index(np.argmax(dst), dst.shape)
            u, v = un[i], un[j]

        branches.append({
            "node_u": node_to_id[tuple(u)],
            "node_v": node_to_id[tuple(v)],
            "voxels": coords
        })
    return branches, node_positions


# ---------------------------------------------------------
# Build graph with radius, curvature, tortuosity, and edge labels
# ---------------------------------------------------------
def skeleton_array_to_graph(skel, seg_full):
    """
    seg_full can be multiclass:
      e.g., 1 = artery, 2 = vein
    We compute edge label by majority vote over branch voxels.
    """
    degree = compute_degree_fast(skel)
    nodes_mask = (degree != 2) & (skel == 1)
    links_mask = (degree == 2) & (skel == 1)


    branches, node_positions = extract_branches_fast(skel, nodes_mask, links_mask)
    branches = compute_branch_features(branches, seg_full)

    edt = distance_transform_edt(seg_full > 0)
    G = nx.Graph()
    node_positions = np.asarray(node_positions, int)

    for i, (z, y, x) in enumerate(node_positions):
      G.add_node(
          i,
          coord_vox=np.array([z, y, x], float),
          degree=int(degree[int(z), int(y), int(x)])
      )

    # Add edges with artery/vein label
    # Assumption: seg_full == 1 -> artery, seg_full == 2 -> vein
    for br in branches:
        u, v = br["node_u"], br["node_v"]
        vox = br["voxels"]
        vals = seg_full[vox[:, 0], vox[:, 1], vox[:, 2]]

        artery_votes = np.sum(vals == 1)
        vein_votes = np.sum(vals == 2)

        if artery_votes == 0 and vein_votes == 0:
            edge_label = -1  # unknown / unlabeled
        else:
            edge_label = 1 if artery_votes > vein_votes else 0  # 1=artery,0=vein

        G.add_edge(
            u, v,
            length_vox=br["length_vox"],
            radius=br["radius"],
            curvature=br["curvature"],
            tortuosity=br["tortuosity"],
            label=edge_label
        )

    G.graph["branches"] = branches

    # Optional PyG data
    pyg_data = None
    if PYG_AVAILABLE:
        mapping = {old: new for new, old in enumerate(G.nodes())}
        G2 = nx.relabel_nodes(G, mapping)

        x = np.array([[G2.nodes[n]["degree"]] for n in G2.nodes()], float)
        pos = np.array([G2.nodes[n]["coord_vox"] for n in G2.nodes()], float)

        edges = np.array(G2.edges(), int)
        if len(edges) > 0:
            edge_index = torch.tensor(edges.T, dtype=torch.long)
            edge_attr = torch.tensor([
                [
                    G2.edges[u, v]["length_vox"],
                    G2.edges[u, v]["radius"],
                    G2.edges[u, v]["curvature"],
                    G2.edges[u, v]["tortuosity"]
                ]
                for (u, v) in edges
            ], dtype=torch.float32)

            edge_labels = torch.tensor(
                [G2.edges[u, v]["label"] for (u, v) in edges],
                dtype=torch.long
            )
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long)
            edge_attr = torch.zeros((0, 4), dtype=torch.float32)
            edge_labels = torch.zeros((0,), dtype=torch.long)

        pyg_data = Data(
            x=torch.tensor(x, dtype=torch.float32),
            pos=torch.tensor(pos, dtype=torch.float32),
            edge_index=edge_index,
            edge_attr=edge_attr,
            edge_label=edge_labels,
        )

    return G, pyg_data


# ---------------------------------------------------------
# Reconstruct volumes from graph
# ---------------------------------------------------------
def reconstruct_artery_vein_volumes(G, shape):
    """
    Reconstruct artery / vein / unknown volumes from the graph edges and
    stored branch voxel coordinates.

    shape: (Z, Y, X)
    Returns:
        artery_vol  (0/1)
        vein_vol    (0/1)
        unknown_vol (0/1)
        combined_vol (0/1/2/3)  e.g. 1=artery, 2=vein, 3=unknown
    """
    artery_vol = np.zeros(shape, dtype=np.uint8)
    vein_vol = np.zeros(shape, dtype=np.uint8)
    unknown_vol = np.zeros(shape, dtype=np.uint8)

    branches = G.graph.get("branches", [])

    for br in branches:
        u = br["node_u"]
        v = br["node_v"]
        if not G.has_edge(u, v):
            continue

        label = G.edges[u, v]["label"]  # 1=artery,0=vein,-1=unknown
        vox = br["voxels"]  # (N,3) array of (z,y,x)

        z = vox[:, 0]
        y = vox[:, 1]
        x = vox[:, 2]

        if label == 1:
            artery_vol[z, y, x] = 1
        elif label == 0:
            vein_vol[z, y, x] = 1
        else:
            unknown_vol[z, y, x] = 1

    combined_vol = np.zeros(shape, dtype=np.uint8)
    combined_vol[artery_vol == 1] = 1
    combined_vol[vein_vol == 1] = 2
    combined_vol[unknown_vol == 1] = 3

    return artery_vol, vein_vol, unknown_vol, combined_vol


# ---------------------------------------------------------
# Main
# ---------------------------------------------------------
def main(input_seg_path, graph_out_path):
    seg_img = nib.load(input_seg_path)
    seg = seg_img.get_fdata()
    affine = seg_img.affine

    # Here seg can be multiclass (1=artery,2=vein,0=background)
    seg_bin = (seg > 0).astype(np.uint8)
    skel = skeletonize_3d_safe(seg_bin)

    #nib.save(nib.Nifti1Image(skel, affine), skeleton_out_path)
    #print("Skeleton saved to", skeleton_out_path)

    G, pyg_data = skeleton_array_to_graph(skel, seg)
    print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    with open(graph_out_path, "wb") as f:
        pickle.dump(G, f)
    print("Graph saved to", graph_out_path)

    shape = seg_bin.shape
    #artery_vol, vein_vol, unknown_vol, combined_vol = reconstruct_artery_vein_volumes(G, shape)

    # Derive base from input_seg_path (without .nii.gz)
    if input_seg_path.endswith(".nii.gz"):
        base = input_seg_path[:-7]
    else:
        base, _ = os.path.splitext(input_seg_path)

    # Save artery/vein/combined volumes
    #nib.save(nib.Nifti1Image(artery_vol, affine), base + "_arteries.nii.gz")
    #nib.save(nib.Nifti1Image(vein_vol, affine),   base + "_veins.nii.gz")
    #nib.save(nib.Nifti1Image(combined_vol, affine), base + "_artery_vein_combined.nii.gz")
    print("Artery/vein volumes saved with base", base)

    # Optionally: save the binary reconstruction
    #nib.save(nib.Nifti1Image(seg_bin, affine), recon_out_path)
    #print("Reconstructed segmentation saved to", recon_out_path)


# ---------------------------------------------------------
# Run
# ---------------------------------------------------------
if __name__ == "__main__":
    input_path = "/projectnb/ec500kb/projects/Project_4/Graph_Creations/Binary_Masks_Predictions/"
    for i in os.listdir(input_path):
        output_mask_path = "/projectnb/ec500kb/projects/Project_4/Graph_Creations/Graphs_from_FineTuned_VesselFM/finetune_graph" + i[-10:-7] + ".gpickle"
        in_path = input_path + i
        print(i)
        if os.path.exists(output_mask_path):
            print(f"Skipping {output_mask_path} → Output already exists.")
            continue
        main(in_path, output_mask_path)
        '''input_path = "/content/case_001.nii.gz"
        base = input_path.replace(".nii.gz", "")
        skeleton_path = base + "_skel.nii.gz"
        graph_path = base + "_graph.gpickle"
        recon_path = base + "_recon.nii.gz"
        #main(input_path, skeleton_path, graph_path, recon_path)'''

predicted_binary_mask205.nii.gz
Skipping /projectnb/ec500kb/projects/Project_4/Graph_Creations/Graphs_from_FineTuned_VesselFM/finetune_graph205.gpickle → Output already exists.
predicted_binary_mask116.nii.gz
Skipping /projectnb/ec500kb/projects/Project_4/Graph_Creations/Graphs_from_FineTuned_VesselFM/finetune_graph116.gpickle → Output already exists.
predicted_binary_mask216.nii.gz
Skipping /projectnb/ec500kb/projects/Project_4/Graph_Creations/Graphs_from_FineTuned_VesselFM/finetune_graph216.gpickle → Output already exists.
predicted_binary_mask022.nii.gz
Skipping /projectnb/ec500kb/projects/Project_4/Graph_Creations/Graphs_from_FineTuned_VesselFM/finetune_graph022.gpickle → Output already exists.
predicted_binary_mask045.nii.gz
Skipping /projectnb/ec500kb/projects/Project_4/Graph_Creations/Graphs_from_FineTuned_VesselFM/finetune_graph045.gpickle → Output already exists.
predicted_binary_mask191.nii.gz
Skipping /projectnb/ec500kb/projects/Project_4/Graph_Creations/Graphs_from_FineTune

/scratch/2058566.1.f/ipykernel_1572888/1097752124.py:26: FutureWarning: `skeletonize_3d` is deprecated since version 0.23 and will be removed in version 0.25. Use `skimage.morphology.skeletonize` instead.
  return skeletonize_3d(vol.astype(bool)).astype(np.uint8)


Graph: 3832 nodes, 1605 edges
Graph saved to /projectnb/ec500kb/projects/Project_4/Graph_Creations/Graphs_from_FineTuned_VesselFM/finetune_graph073.gpickle
Artery/vein volumes saved with base /projectnb/ec500kb/projects/Project_4/Graph_Creations/Binary_Masks_Predictions/predicted_binary_mask073
predicted_binary_mask238.nii.gz
Graph: 2628 nodes, 1268 edges
Graph saved to /projectnb/ec500kb/projects/Project_4/Graph_Creations/Graphs_from_FineTuned_VesselFM/finetune_graph238.gpickle
Artery/vein volumes saved with base /projectnb/ec500kb/projects/Project_4/Graph_Creations/Binary_Masks_Predictions/predicted_binary_mask238
predicted_binary_mask173.nii.gz
Graph: 3449 nodes, 872 edges
Graph saved to /projectnb/ec500kb/projects/Project_4/Graph_Creations/Graphs_from_FineTuned_VesselFM/finetune_graph173.gpickle
Artery/vein volumes saved with base /projectnb/ec500kb/projects/Project_4/Graph_Creations/Binary_Masks_Predictions/predicted_binary_mask173
predicted_binary_mask134.nii.gz
Graph: 21249 node

In [2]:


with open(graph_path, "rb") as f:
    G = pickle.load(f)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
import numpy as np
degrees = [G.nodes[n]["degree"] for n in G.nodes()]
print("Degree histogram:", np.bincount(degrees))
#list(G.nodes(data=True))[:5]
list(G.edges(data=True))[:5]


Graph: 89573 nodes, 1540 edges
Degree histogram: [    3   556     0  3852  8410 16126 10885 12226 33147  2163  1184   655
   223    87    43    10     2     1]


[(12,
  12,
  {'length_vox': 2.0,
   'radius': 1.0,
   'curvature': 0.0,
   'tortuosity': 1.9999999800000003,
   'label': 1}),
 (13,
  27,
  {'length_vox': 1.0,
   'radius': 1.0,
   'curvature': 0.0,
   'tortuosity': 1.0,
   'label': 1}),
 (26,
  38,
  {'length_vox': 1.0,
   'radius': 1.0,
   'curvature': 0.0,
   'tortuosity': 1.0,
   'label': 0}),
 (67,
  86,
  {'length_vox': 1.0,
   'radius': 1.0,
   'curvature': 0.0,
   'tortuosity': 1.0,
   'label': 1}),
 (111,
  134,
  {'length_vox': 1.0,
   'radius': 1.0,
   'curvature': 0.0,
   'tortuosity': 1.0,
   'label': 0})]

In [20]:
import numpy as np

def reconstruct_artery_vein_from_graph(G, shape, spacing=(1,1,1)):
    """
    Pure graph-based vessel reconstruction.
    Uses ONLY:
        - branch skeleton voxel coordinates
        - per-edge radius (br["radius"])
        - per-edge label (artery / vein / unknown)

    NO original segmentation
    NO node radii

    Returns:
        artery_vol, vein_vol, unknown_vol, combined_vol
    """
    Z, Y, X = shape

    artery_vol  = np.zeros(shape, dtype=np.uint8)
    vein_vol    = np.zeros(shape, dtype=np.uint8)
    unknown_vol = np.zeros(shape, dtype=np.uint8)

    sz, sy, sx = spacing

    for br in G.graph.get("branches", []):
        vox = br["voxels"]          # (N,3) array
        u, v = br["node_u"], br["node_v"]

        label  = G.edges[u, v]["label"]      # 1=artery, 0=vein, -1=unknown
        R_edge = float(G.edges[u, v]["radius"])

        for (z, y, x) in vox:
            z = int(z); y = int(y); x = int(x)

            # Compute voxel extent from radius
            rz = int(np.ceil(R_edge / sz))
            ry = int(np.ceil(R_edge / sy))
            rx = int(np.ceil(R_edge / sx))

            # Bounding box
            z0, z1 = max(0, z-rz), min(Z, z+rz+1)
            y0, y1 = max(0, y-ry), min(Y, y+ry+1)
            x0, x1 = max(0, x-rx), min(X, x+rx+1)

            # Create spherical mask in physical space
            zz, yy, xx = np.ogrid[z0:z1, y0:y1, x0:x1]
            sphere = (
                ((zz - z)*sz)**2 +
                ((yy - y)*sy)**2 +
                ((xx - x)*sx)**2
            ) <= R_edge**2

            # Paint sphere according to label
            if label == 1:
                artery_vol[z0:z1, y0:y1, x0:x1][sphere] = 1
            elif label == 0:
                vein_vol[z0:z1, y0:y1, x0:x1][sphere]   = 1
            else:
                unknown_vol[z0:z1, y0:y1, x0:x1][sphere] = 1

    # Combined result
    combined = np.zeros(shape, dtype=np.uint8)
    combined[artery_vol == 1]  = 1
    combined[vein_vol == 1]    = 2
    combined[unknown_vol == 1] = 3

    return artery_vol, vein_vol, unknown_vol, combined


In [21]:
# Load graph
import pickle
import nibabel as nib
with open("guess001_graph.gpickle", "rb") as f:
    G = pickle.load(f)

# Load original segmentation for shape + spacing
nii = nib.load("/projectnb/ec500kb/projects/Fall_2025_Projects/Project_4_VesselFM/data/nnUNet_preprocessed/Dataset001_nnunet/gt_segmentations/case_001.nii.gz")
shape   = nii.shape
affine  = nii.affine
spacing = nii.header.get_zooms()[:3]

artery, vein, unknown, combined = reconstruct_artery_vein_from_graph(G, shape, spacing)

# Save results
#nib.save(nib.Nifti1Image(artery,  affine), "case_001_artery_reconstructed.nii.gz")
#nib.save(nib.Nifti1Image(vein,    affine), "case_001_vein_reconstructed.nii.gz")
#nib.save(nib.Nifti1Image(unknown, affine), "case_001_unknown_reconstructed.nii.gz")
nib.save(nib.Nifti1Image(combined, affine), "case_001_combined_reconstructed.nii.gz")

print("Artery / Vein segmentation reconstructed successfully.")


Artery / Vein segmentation reconstructed successfully.


In [22]:
def propagate_nearest_label(seg, recon):
    """
    Fill seg>=1 region using nearest NONZERO voxel from recon.
    recon is typically your artery/vein skeleton.
    """

    seg    = seg.astype(np.int32)
    recon  = recon.astype(np.int32)

    # Region to fill
    region_mask = seg > 0

    # Valid labeled seeds in recon
    seed_mask = recon > 0

    # Nearest labeled voxel in recon
    _, nearest_idx = distance_transform_edt(
        ~seed_mask,
        return_indices=True
    )

    nz, ny, nx = nearest_idx

    # Get the label at the nearest recon voxel
    nearest_values = recon[nz, ny, nx]

    # Output: only fill seg>=1 region
    out = np.zeros_like(seg, dtype=np.int32)
    out[region_mask] = nearest_values[region_mask]

    return out

In [23]:
import pickle
import sys
import os
import numpy as np
import nibabel as nib
import networkx as nx
from scipy.ndimage import convolve, label, distance_transform_edt

nii = nib.load("/projectnb/ec500kb/projects/Fall_2025_Projects/Project_4_VesselFM/data/nnUNet_preprocessed/Dataset001_nnunet/gt_segmentations/case_001.nii.gz")
seg = nii.get_fdata().astype(np.int32)

recon = combined   # your artery/vein from graph

out = propagate_nearest_label(seg, recon)

nib.save(nib.Nifti1Image(out, nii.affine), "nearest_label_propagated.nii.gz")


In [24]:
import nibabel as nib
import numpy as np

# Load ground truth & reconstruction
gt_img = nib.load("/projectnb/ec500kb/projects/Fall_2025_Projects/Project_4_VesselFM/data/nnUNet_preprocessed/Dataset001_nnunet/gt_segmentations/case_001.nii.gz")
recon_img = nib.load("nearest_label_propagated.nii.gz")

gt = gt_img.get_fdata().astype(int)
recon = recon_img.get_fdata().astype(int)

# Class names
classes = {
    0: "background",
    1: "artery",
    2: "vein",
    3: "unknown"
}

# Count voxels per class
counts_gt = {c: int(np.sum(gt == c)) for c in classes}
counts_recon = {c: int(np.sum(recon == c)) for c in classes}

print("=== Voxel Counts ===")
print("Ground Truth:", counts_gt)
print("Reconstruction:", counts_recon)

# Compute confusion matrix
cm = np.zeros((4,4), dtype=int)
for c in range(4):
    for r in range(4):
        cm[c, r] = np.sum((gt == c) & (recon == r))

print("\n=== Confusion Matrix (rows = GT, cols = Reconstruction) ===")
print("Rows:    0 BG, 1 Art, 2 Vein, 3 Unk")
print("Columns: 0 BG, 1 Art, 2 Vein, 3 Unk\n")
print(cm)

# Dice coefficient for artery & vein
def dice(a, b):
    intersection = np.sum((a==1) & (b==1))
    return 2*intersection / (np.sum(a==1) + np.sum(b==1) + 1e-8)

dice_artery = dice(gt==1, recon==1)
dice_vein   = dice(gt==2, recon==2)

print("\n=== Dice Scores ===")
print("Artery Dice:", dice_artery)
print("Vein Dice:  ", dice_vein)


=== Voxel Counts ===
Ground Truth: {0: 66872616, 1: 349106, 2: 411430, 3: 0}
Reconstruction: {0: 66872616, 1: 379566, 2: 380970, 3: 0}

=== Confusion Matrix (rows = GT, cols = Reconstruction) ===
Rows:    0 BG, 1 Art, 2 Vein, 3 Unk
Columns: 0 BG, 1 Art, 2 Vein, 3 Unk

[[66872616        0        0        0]
 [       0   264582    84524        0]
 [       0   114984   296446        0]
 [       0        0        0        0]]

=== Dice Scores ===
Artery Dice: 0.7262032848798811
Vein Dice:   0.7482231196365378
